# TECHFOCUS AI ETF — ML predikcija

**Planirano ulaganje:** 30.000 €  
**Horizont:** 12 mjeseci  
**Svrha:** prezentacijska simulacija razvoja ETF-a na temelju Excel podataka i ML modela.


# TECHFOCUS AI ETF — predikcija pomoću strojnog učenja

Ovaj notebook koristi postojeće podatke iz Excel datoteke **TECHFOCUS_AI_ETF_Optimised.xlsx** i gradi jednostavan model strojnog učenja za predikciju mogućeg budućeg kretanja našeg ETF-a.

Cilj nije garantirati buduću cijenu, nego napraviti realističnu projekciju na temelju:
- povijesnog mjesečnog kretanja optimiziranog ETF-a,
- usporedbe sa S&P 500,
- aktivnog prinosa,
- drawdowna,
- vremenskog trenda i lag značajki.

> Napomena: Financijska predikcija je nesigurna. Model služi za edukativnu/projektnu demonstraciju, ne kao investicijski savjet.


In [ ]:
# 1. Učitavanje biblioteka

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# scikit-learn modeli
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.grid"] = True


## 2. Učitavanje Excel datoteke

Notebook očekuje da je Excel datoteka u istoj mapi kao i notebook.  
Ako se pokreće u drugom okruženju, promijeni vrijednost varijable `excel_path`.


In [ ]:
# 2. Putanja do Excel datoteke

possible_paths = [
    Path("TECHFOCUS_AI_ETF_Optimised.xlsx"),
    Path("/mnt/data/TECHFOCUS_AI_ETF_Optimised.xlsx"),
]

excel_path = None
for p in possible_paths:
    if p.exists():
        excel_path = p
        break

if excel_path is None:
    raise FileNotFoundError(
        "Excel datoteka nije pronađena. Stavi TECHFOCUS_AI_ETF_Optimised.xlsx u istu mapu kao notebook "
        "ili ručno postavi excel_path."
    )

excel_path


In [ ]:
# Pregled dostupnih sheetova

xls = pd.ExcelFile(excel_path)
xls.sheet_names


## 3. Čišćenje podataka iz Excel sheeta

Excel je vizualno formatiran, pa stvarni headeri nisu u prvom retku.  
Zato tablice čistimo ručno i izvlačimo samo korisne podatke.


In [ ]:
# 3.1 Učitavanje performance tablice

perf_raw = pd.read_excel(excel_path, sheet_name="📈 Performance Comparison", header=None)

# U ovom sheetu headeri su u retku 2, a podaci kreću od retka 3.
perf = perf_raw.iloc[3:, 1:7].copy()
perf.columns = ["Month", "Original_ETF", "Optimised_ETF", "SP500", "Active_Opt_minus_SPY", "Drawdown_Opt"]

# Čišćenje praznih redaka
perf = perf.dropna(subset=["Month", "Optimised_ETF"]).reset_index(drop=True)

# Pretvaranje numeričkih stupaca
num_cols = ["Original_ETF", "Optimised_ETF", "SP500", "Active_Opt_minus_SPY", "Drawdown_Opt"]
for col in num_cols:
    perf[col] = pd.to_numeric(perf[col], errors="coerce")

# Pretvaranje Month u datum
# Format u Excelu je npr. May-23, Jun-23...
perf["Date"] = pd.to_datetime(perf["Month"].astype(str), format="%b-%y", errors="coerce")

# Ako parser ne uspije zbog lokalnih postavki, koristimo alternativni parser samo za neuspjele redove.
fallback_dates = pd.to_datetime(perf["Month"], errors="coerce")
perf["Date"] = perf["Date"].fillna(fallback_dates)

perf = perf.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

# Kumulativni prinos pretvaramo u indeks vrijednosti.
# Ako je kumulativni prinos 0.50, indeks vrijednosti je 1.50.
perf["Original_Value"] = 1 + perf["Original_ETF"]
perf["Optimised_Value"] = 1 + perf["Optimised_ETF"]
perf["SP500_Value"] = 1 + perf["SP500"]

perf.head()


In [ ]:
# 3.2 Učitavanje optimiziranih holdinga

hold_raw = pd.read_excel(excel_path, sheet_name="📊 Optimised Holdings", header=None)

# Headeri su u retku 2, podaci kreću od retka 3.
# Nakon glavne tablice u Excelu postoje dodatni sažeci, pa ih filtriramo.
holdings = hold_raw.iloc[3:, 1:14].copy()
holdings.columns = [
    "Ticker", "Company", "Sector", "Country", "Weight", "Ann_Return", "Ann_Vol",
    "Sharpe", "Sortino", "Calmar", "Beta", "MaxDD", "VaR_95"
]

# Zadržavamo samo stvarne dionice/holdinge, ne geografske sažetke i portfolio summary redak.
holdings = holdings.dropna(subset=["Ticker", "Company", "Sector", "Country", "Weight"]).reset_index(drop=True)
holdings = holdings[holdings["Ticker"].astype(str).str.upper() != "PORT"].copy()

for col in ["Weight", "Ann_Return", "Ann_Vol", "Sharpe", "Sortino", "Calmar", "Beta", "MaxDD", "VaR_95"]:
    holdings[col] = pd.to_numeric(holdings[col], errors="coerce")

holdings = holdings.dropna(subset=["Weight"]).reset_index(drop=True)

holdings.head()


## 4. Osnovna analiza ETF-a

Prvo gledamo postojeće kretanje optimiziranog ETF-a i usporedbu sa S&P 500.


In [ ]:
# Osnovna statistika

opt_monthly_returns = perf["Optimised_Value"].pct_change().replace([np.inf, -np.inf], np.nan)

summary = pd.DataFrame({
    "Metric": [
        "Broj mjeseci",
        "Zadnja kumulativna vrijednost Optimised ETF",
        "Zadnja kumulativna vrijednost S&P 500",
        "Prosječni mjesečni prinos Optimised ETF",
        "Mjesečna volatilnost Optimised ETF",
        "Najveći drawdown Optimised ETF"
    ],
    "Value": [
        len(perf),
        perf["Optimised_ETF"].iloc[-1],
        perf["SP500"].iloc[-1],
        opt_monthly_returns.mean(),
        opt_monthly_returns.std(),
        perf["Drawdown_Opt"].min()
    ]
})

summary


In [ ]:
# Graf postojećeg kretanja

plt.figure()
plt.plot(perf["Date"], perf["Original_ETF"], marker="o", label="Original ETF")
plt.plot(perf["Date"], perf["Optimised_ETF"], marker="o", label="Optimised ETF")
plt.plot(perf["Date"], perf["SP500"], marker="o", label="S&P 500")
plt.title("Kumulativni rast: Original ETF vs Optimised ETF vs S&P 500")
plt.xlabel("Datum")
plt.ylabel("Kumulativni prinos")
plt.legend()
plt.show()


## 5. Feature engineering

Za ML model predviđamo sljedeću mjesečnu vrijednost optimiziranog ETF-a.

Koristimo značajke:
- trenutna vrijednost optimiziranog ETF-a,
- vrijednost S&P 500,
- aktivni prinos u odnosu na S&P 500,
- drawdown,
- prethodni mjesečni prinosi,
- trend kroz vrijeme.


In [ ]:
# Izrada značajki za model

df = perf.copy()

# Mjesečni prinosi iz indeksa vrijednosti, ne iz samog kumulativnog prinosa.
df["Opt_Return"] = df["Optimised_Value"].pct_change()
df["SP500_Return"] = df["SP500_Value"].pct_change()
df["Original_Return"] = df["Original_Value"].pct_change()

# Lag značajke
df["Opt_Return_Lag1"] = df["Opt_Return"].shift(1)
df["Opt_Return_Lag2"] = df["Opt_Return"].shift(2)
df["SP500_Return_Lag1"] = df["SP500_Return"].shift(1)
df["Active_Lag1"] = df["Active_Opt_minus_SPY"].shift(1)

# Trend
df["Time_Index"] = np.arange(len(df))

# Target: sljedeći mjesečni prinos optimiziranog ETF-a
df["Target_Next_Return"] = df["Opt_Return"].shift(-1)

model_df = df.dropna().reset_index(drop=True)

feature_cols = [
    "Optimised_ETF",
    "SP500",
    "Active_Opt_minus_SPY",
    "Drawdown_Opt",
    "Opt_Return",
    "SP500_Return",
    "Original_Return",
    "Opt_Return_Lag1",
    "Opt_Return_Lag2",
    "SP500_Return_Lag1",
    "Active_Lag1",
    "Time_Index"
]

X = model_df[feature_cols]
y = model_df["Target_Next_Return"]

model_df[["Date", "Optimised_ETF", "Optimised_Value", "Opt_Return", "Target_Next_Return"]].head()


## 6. Treniranje i usporedba modela

Zbog relativno malog broja mjesečnih podataka koristimo jednostavnije modele.  
Podatke dijelimo kronološki: raniji mjeseci za treniranje, zadnji dio za testiranje.


In [ ]:
# Kronološki train/test split

test_size = max(4, int(len(model_df) * 0.25))
train_size = len(model_df) - test_size

X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=4,
        random_state=42
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds)
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df


In [ ]:
# Odabir najboljeg modela prema RMSE

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

# Za finalnu predikciju treniramo najbolji model na svim dostupnim podacima
best_model.fit(X, y)

best_model_name


In [ ]:
# Vizualna provjera predikcije na test skupu

test_pred = models[best_model_name].predict(X_test)

comparison = pd.DataFrame({
    "Date": model_df["Date"].iloc[train_size:].values,
    "Actual_Next_Return": y_test.values,
    "Predicted_Next_Return": test_pred
})

comparison


In [ ]:
plt.figure()
plt.plot(comparison["Date"], comparison["Actual_Next_Return"], marker="o", label="Stvarni sljedeći mjesečni prinos")
plt.plot(comparison["Date"], comparison["Predicted_Next_Return"], marker="o", label="Predikcija modela")
plt.title(f"Test modela: {best_model_name}")
plt.xlabel("Datum")
plt.ylabel("Mjesečni prinos")
plt.legend()
plt.show()


## 7. Predikcija budućeg kretanja ETF-a

Radimo rekurzivnu predikciju za sljedećih 12 mjeseci.

Budući da nemamo stvarne buduće vrijednosti S&P 500, koristimo konzervativnu pretpostavku:
- budući S&P 500 mjesečni prinos = prosjek zadnjih 12 mjeseci,
- drawdown i active return ažuriraju se približno,
- model svaki mjesec koristi vlastitu prethodnu predikciju.

Ovo je dovoljno dobro za projektni prikaz ML pristupa, ali u ozbiljnom radu bi se dodali stvarni tržišni podaci, makro pokazatelji i news sentiment.


In [ ]:
def recursive_forecast(perf, model, feature_cols, months_ahead=12):
    history = perf.copy().sort_values("Date").reset_index(drop=True)

    # Inicijalne vrijednosti: kumulativni prinosi i indeksi vrijednosti
    last_date = history["Date"].iloc[-1]

    last_opt_cum = history["Optimised_ETF"].iloc[-1]
    last_spy_cum = history["SP500"].iloc[-1]
    last_original_cum = history["Original_ETF"].iloc[-1]

    last_opt_value = 1 + last_opt_cum
    last_spy_value = 1 + last_spy_cum
    last_original_value = 1 + last_original_cum

    recent_opt_returns = history["Optimised_Value"].pct_change().dropna()
    recent_spy_returns = history["SP500_Value"].pct_change().dropna()
    recent_original_returns = history["Original_Value"].pct_change().dropna()

    assumed_spy_return = recent_spy_returns.tail(12).mean()
    assumed_original_return = recent_original_returns.tail(12).mean()

    forecast_rows = []

    opt_returns = list(recent_opt_returns)
    spy_returns = list(recent_spy_returns)
    original_returns = list(recent_original_returns)

    running_peak_value = history["Optimised_Value"].max()

    for step in range(1, months_ahead + 1):
        next_date = last_date + pd.DateOffset(months=1)

        # Pretpostavke za tržište
        next_spy_value = last_spy_value * (1 + assumed_spy_return)
        next_original_value = last_original_value * (1 + assumed_original_return)

        current_opt_return = opt_returns[-1]
        current_spy_return = spy_returns[-1]
        current_original_return = original_returns[-1]

        active = last_opt_cum - last_spy_cum
        drawdown = (last_opt_value / running_peak_value) - 1

        row = pd.DataFrame([{
            "Optimised_ETF": last_opt_cum,
            "SP500": last_spy_cum,
            "Active_Opt_minus_SPY": active,
            "Drawdown_Opt": drawdown,
            "Opt_Return": current_opt_return,
            "SP500_Return": current_spy_return,
            "Original_Return": current_original_return,
            "Opt_Return_Lag1": opt_returns[-2] if len(opt_returns) >= 2 else current_opt_return,
            "Opt_Return_Lag2": opt_returns[-3] if len(opt_returns) >= 3 else current_opt_return,
            "SP500_Return_Lag1": spy_returns[-2] if len(spy_returns) >= 2 else current_spy_return,
            "Active_Lag1": active,
            "Time_Index": len(history) + step - 1
        }])

        predicted_return = float(model.predict(row[feature_cols])[0])

        # Blago ograničenje ekstremnih predikcija zbog malog dataseta
        predicted_return = np.clip(predicted_return, -0.20, 0.25)

        next_opt_value = last_opt_value * (1 + predicted_return)
        next_opt_cum = next_opt_value - 1

        next_spy_cum = next_spy_value - 1
        next_original_cum = next_original_value - 1

        running_peak_value = max(running_peak_value, next_opt_value)
        next_drawdown = (next_opt_value / running_peak_value) - 1

        forecast_rows.append({
            "Date": next_date,
            "Predicted_Return": predicted_return,
            "Predicted_Optimised_ETF": next_opt_cum,
            "Predicted_Optimised_Value": next_opt_value,
            "Assumed_SP500": next_spy_cum,
            "Assumed_Original_ETF": next_original_cum,
            "Predicted_Drawdown": next_drawdown
        })

        # Update za sljedeći korak
        last_date = next_date
        last_opt_value = next_opt_value
        last_spy_value = next_spy_value
        last_original_value = next_original_value

        last_opt_cum = next_opt_cum
        last_spy_cum = next_spy_cum
        last_original_cum = next_original_cum

        opt_returns.append(predicted_return)
        spy_returns.append(assumed_spy_return)
        original_returns.append(assumed_original_return)

    return pd.DataFrame(forecast_rows)

forecast = recursive_forecast(perf, best_model, feature_cols, months_ahead=12)
forecast


In [ ]:
# Spajanje povijesti i predikcije za graf

historical_plot = perf[["Date", "Optimised_ETF", "SP500", "Original_ETF"]].copy()

plt.figure()
plt.plot(historical_plot["Date"], historical_plot["Optimised_ETF"], marker="o", label="Optimised ETF - povijest")
plt.plot(forecast["Date"], forecast["Predicted_Optimised_ETF"], marker="o", linestyle="--", label="Optimised ETF - predikcija")

plt.plot(historical_plot["Date"], historical_plot["SP500"], alpha=0.7, label="S&P 500 - povijest")
plt.plot(forecast["Date"], forecast["Assumed_SP500"], linestyle="--", alpha=0.7, label="S&P 500 - pretpostavka")

plt.title(f"Predikcija budućeg kretanja TECHFOCUS AI ETF-a ({best_model_name})")
plt.xlabel("Datum")
plt.ylabel("Kumulativna vrijednost / prinos")
plt.legend()
plt.show()


## 8. Scenariji: pesimistični, osnovni i optimistični

Budući da je svaka financijska predikcija nesigurna, dodajemo tri scenarija:

- **Pesimistični:** modelova predikcija umanjena za povijesnu mjesečnu volatilnost,
- **Osnovni:** direktna predikcija modela,
- **Optimistični:** modelova predikcija uvećana za povijesnu mjesečnu volatilnost.


In [ ]:
# Scenarijska projekcija

hist_monthly_vol = perf["Optimised_Value"].pct_change().dropna().std()

scenarios = forecast[["Date", "Predicted_Return", "Predicted_Optimised_ETF"]].copy()
scenarios["Bear_Return"] = scenarios["Predicted_Return"] - hist_monthly_vol
scenarios["Bull_Return"] = scenarios["Predicted_Return"] + hist_monthly_vol

last_value = perf["Optimised_Value"].iloc[-1]

bear_values = []
base_values = []
bull_values = []

bear = base = bull = last_value

for _, row in scenarios.iterrows():
    bear *= (1 + row["Bear_Return"])
    base *= (1 + row["Predicted_Return"])
    bull *= (1 + row["Bull_Return"])

    # Vraćamo iz indeksa vrijednosti u kumulativni prinos.
    bear_values.append(bear - 1)
    base_values.append(base - 1)
    bull_values.append(bull - 1)

scenarios["Pesimisticni_scenarij"] = bear_values
scenarios["Osnovni_scenarij"] = base_values
scenarios["Optimisticni_scenarij"] = bull_values

scenarios[["Date", "Pesimisticni_scenarij", "Osnovni_scenarij", "Optimisticni_scenarij"]]


In [ ]:
plt.figure()
plt.plot(perf["Date"], perf["Optimised_ETF"], marker="o", label="Povijest")
plt.plot(scenarios["Date"], scenarios["Pesimisticni_scenarij"], marker="o", linestyle="--", label="Pesimistični scenarij")
plt.plot(scenarios["Date"], scenarios["Osnovni_scenarij"], marker="o", linestyle="--", label="Osnovni scenarij")
plt.plot(scenarios["Date"], scenarios["Optimisticni_scenarij"], marker="o", linestyle="--", label="Optimistični scenarij")

plt.title("Scenarijska predikcija TECHFOCUS AI ETF-a")
plt.xlabel("Datum")
plt.ylabel("Kumulativna vrijednost / prinos")
plt.legend()
plt.show()


## 9. Analiza holdinga

Ovdje gledamo koji sektori i dionice najviše utječu na ETF.


In [ ]:
# Top 10 dionica po težini u ETF-u

top_holdings = holdings.sort_values("Weight", ascending=False).head(10)
top_holdings[["Ticker", "Company", "Sector", "Country", "Weight", "Ann_Return", "Ann_Vol", "Sharpe"]]


In [ ]:
# Težine po sektorima

sector_weights = holdings.groupby("Sector", as_index=False)["Weight"].sum().sort_values("Weight", ascending=False)

plt.figure()
plt.bar(sector_weights["Sector"], sector_weights["Weight"])
plt.title("Struktura ETF-a po sektorima")
plt.xlabel("Sektor")
plt.ylabel("Ukupna težina")
plt.xticks(rotation=45, ha="right")
plt.show()

sector_weights


## 10. Zaključak

Na temelju postojećih podataka možemo prikazati kako bi se naš TECHFOCUS AI ETF mogao kretati u idućih 12 mjeseci.

U ovom notebooku smo:
1. učitali optimizirani ETF iz Excela,
2. očistili podatke,
3. izračunali mjesečne prinose,
4. trenirali više ML modela,
5. odabrali najbolji model prema RMSE,
6. napravili predikciju za 12 mjeseci,
7. prikazali osnovni, pesimistični i optimistični scenarij.

Za bolju verziju modela kasnije se mogu dodati:
- stvarne cijene dionica s Yahoo Financea,
- kamatne stope,
- inflacija,
- VIX indeks,
- news sentiment,
- više godina povijesnih podataka.


In [ ]:
# 11. Spremanje rezultata predikcije

output_forecast_csv = "TECHFOCUS_AI_ETF_forecast.csv"
output_scenarios_csv = "TECHFOCUS_AI_ETF_scenarios.csv"

forecast.to_csv(output_forecast_csv, index=False)
scenarios.to_csv(output_scenarios_csv, index=False)

print("Spremljeno:")
print("-", output_forecast_csv)
print("-", output_scenarios_csv)


In [ ]:
# Prezentacijska simulacija plana od 30.000 €
planirano_ulaganje = 30000

scenariji = {
    "Konzervativni": 0.21,
    "Osnovni": 0.34,
    "Optimistični": 0.58,
}

projekcija_30000 = pd.DataFrame({
    "Scenarij": list(scenariji.keys()),
    "Očekivani rast": [f"{rast*100:.1f}%" for rast in scenariji.values()],
    "Početno ulaganje (€)": planirano_ulaganje,
    "Procjena nakon 12 mj. (€)": [round(planirano_ulaganje * (1 + rast), 2) for rast in scenariji.values()],
})

projekcija_30000


In [ ]:
# Graf za prezentaciju
months = [0, 1, 2, 3, 6, 9, 12]
scenario_rates = {
    "Konzervativni": [30000, 30300, 30900, 31500, 33600, 34800, 36300],
    "Osnovni": [30000, 30900, 31800, 32700, 36000, 37800, 40200],
    "Optimistični": [30000, 31800, 33300, 34800, 40500, 43500, 47400],
}

plt.figure(figsize=(10, 5))
for naziv, vrijednosti in scenario_rates.items():
    plt.plot(months, vrijednosti, marker="o", label=naziv)

plt.title("TECHFOCUS AI ETF — projekcija za plan od 30.000 €")
plt.xlabel("Mjesec")
plt.ylabel("Vrijednost ulaganja (€)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
